## Setup

In [ ]:
# Imports
import sys
sys.path.append('..')

from data.dataset import DocumentDataset
from src.loader import load_and_clean_pdf
from src.chunker import TextChunker
from src.summarizer import DocumentSummarizer
from src.evaluator import SummaryEvaluator, calculate_faithfulness_score

import warnings
warnings.filterwarnings('ignore')

print(" Imports successful!")

/Users/pakhichatterjee/git/long-doc-summarization-nlp/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful!


## 1. Explore the Dataset

In [ ]:
# Load dataset
dataset = DocumentDataset()

print(f"Loaded {len(dataset)} documents\n")
print("="*80)

# Display statistics
stats = dataset.get_statistics()
print("\n Dataset Statistics:")
print(f"  Total documents: {stats['total_documents']}")
print(f"  Total words: {stats['total_words']:,}")
print(f"  Average words: {stats['avg_words']:,.0f}")
print(f"  Range: {stats['min_words']:,} - {stats['max_words']:,} words")

In [ ]:
# List all documents
print("\nDocuments in Dataset:\n")
print("="*80)

for i, doc_info in enumerate(dataset.list_documents(), 1):
    print(f"\n{i}. [{doc_info['id']}]")
    print(f"   Title: {doc_info['title'][:70]}...")
    print(f"   Words: {doc_info['word_count']:,}")
    if doc_info['arxiv_id']:
        print(f"   arXiv: {doc_info['arxiv_id']}")

In [ ]:
# View a specific document
doc = dataset[0]  # Change index to view different documents

print(f"\nDocument: {doc['id']}")
print("="*80)
print(f"Title: {doc['metadata']['title']}")
print(f"Words: {doc['metadata']['word_count']:,}")
print(f"\nFirst 500 characters:")
print("-"*80)
print(doc['text'][:500])
print("-"*80)

## 2. Test Summarization

In [ ]:
# Initialize components
print("Initializing summarizer...")
summarizer = DocumentSummarizer(
    model_name="facebook/bart-large-cnn",
    max_length=150,
    min_length=50
)

chunker = TextChunker(chunk_size=1024, overlap=128)

print("Ready to summarize!")

In [ ]:
# Select a document to summarize
doc_index = 0  # Change this to test different documents
doc = dataset[doc_index]

print(f"\n Summarizing: {doc['id']}")
print(f"Title: {doc['metadata']['title'][:70]}...")
print(f"Length: {doc['metadata']['word_count']:,} words")
print("="*80)

In [ ]:
# Step 1: Chunk the document
print("\n1. Chunking document...")
chunks = chunker.chunk_text(doc['text'], method="sentences")
print(f" Created {len(chunks)} chunks")

# Show chunk sizes
chunk_sizes = [len(c.split()) for c in chunks]
print(f"   Chunk sizes: min={min(chunk_sizes)}, max={max(chunk_sizes)}, avg={sum(chunk_sizes)/len(chunk_sizes):.0f} words")

In [ ]:
# Step 2: Hierarchical summarization
print("\n2. Generating hierarchical summary...")
summary = summarizer.hierarchical_summarize(chunks)

print(f"\n Summary ({len(summary.split())} words):")
print("="*80)
print(summary)
print("="*80)

In [ ]:
# Step 3: Evaluate the summary
print("\n3. Evaluating summary quality...")

faithfulness = calculate_faithfulness_score(doc['text'], summary)
compression = len(summary.split()) / len(doc['text'].split())

print(f"\n Metrics:")
print(f"   Compression ratio: {compression:.1%}")
print(f"   Faithfulness score: {faithfulness:.1%}")
print(f"   Original: {len(doc['text'].split()):,} words")
print(f"   Summary: {len(summary.split())} words")

## 3. Compare Methods

In [ ]:
# Compare hierarchical vs concatenate methods
print("\nComparing summarization methods...\n")
print("="*80)

# Use the same document and chunks
methods = ['hierarchical', 'concatenate']
results = {}

for method in methods:
    print(f"\nMethod: {method}")
    print("-"*80)
    
    if method == 'hierarchical':
        summary = summarizer.hierarchical_summarize(chunks)
    else:
        summary = summarizer.concatenate_and_summarize(chunks)
    
    faithfulness = calculate_faithfulness_score(doc['text'], summary)
    compression = len(summary.split()) / len(doc['text'].split())
    
    results[method] = {
        'summary': summary,
        'faithfulness': faithfulness,
        'compression': compression,
        'words': len(summary.split())
    }
    
    print(f"Summary: {summary[:150]}...")
    print(f"Words: {results[method]['words']}")
    print(f"Compression: {compression:.1%}")
    print(f"Faithfulness: {faithfulness:.1%}")

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

methods = list(results.keys())
compressions = [results[m]['compression'] for m in methods]
faithfulness = [results[m]['faithfulness'] for m in methods]

axes[0].bar(methods, compressions, color='skyblue', alpha=0.7)
axes[0].set_title('Compression Ratio')
axes[0].set_ylabel('Ratio')

axes[1].bar(methods, faithfulness, color='lightgreen', alpha=0.7)
axes[1].set_title('Faithfulness Score')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

## 4. Batch Processing

In [ ]:
# Process multiple documents
num_docs = 3  # Change to process more documents

print(f"\nProcessing {num_docs} documents...\n")
print("="*80)

batch_results = []

for i in range(min(num_docs, len(dataset))):
    doc = dataset[i]
    print(f"\n[{i+1}/{num_docs}] {doc['id']}")
    
    # Chunk and summarize
    chunks = chunker.chunk_text(doc['text'], method="sentences")
    summary = summarizer.hierarchical_summarize(chunks)
    
    # Metrics
    faithfulness = calculate_faithfulness_score(doc['text'], summary)
    compression = len(summary.split()) / len(doc['text'].split())
    
    batch_results.append({
        'doc_id': doc['id'],
        'title': doc['metadata']['title'][:50],
        'words': len(summary.split()),
        'compression': compression,
        'faithfulness': faithfulness,
        'summary': summary
    })
    
    print(f"  Compression: {compression:.1%} | Faithfulness: {faithfulness:.1%}")

print("\n" + "="*80)
print(" Batch processing complete!")

In [ ]:
# Visualize batch results
import pandas as pd

df = pd.DataFrame(batch_results)
print("\n Batch Results Summary:")
print("="*80)
print(df[['doc_id', 'words', 'compression', 'faithfulness']])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.plot(x='doc_id', y='compression', kind='bar', ax=axes[0], color='skyblue', alpha=0.7)
axes[0].set_title('Compression Ratio by Document')
axes[0].set_ylabel('Compression Ratio')
axes[0].tick_params(axis='x', rotation=45)

df.plot(x='doc_id', y='faithfulness', kind='bar', ax=axes[1], color='lightgreen', alpha=0.7)
axes[1].set_title('Faithfulness Score by Document')
axes[1].set_ylabel('Faithfulness Score')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Save Your Work

In [ ]:
# Save results to JSON
import json
from pathlib import Path

output_dir = Path('..') / 'notebook_results'
output_dir.mkdir(exist_ok=True)

output_file = output_dir / 'exploration_results.json'
with open(output_file, 'w') as f:
    json.dump(batch_results, f, indent=2)

print(f" Results saved to {output_file}")